<a href="https://colab.research.google.com/github/Ravi-ranjan1801/CUDA-Lab/blob/main/cuda_lab_mid_find_max%26min_of_vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

implement a cuda c program to find max and min element of a vector, take size of vector as input

In [ ]:
%%writefile max_min.cu
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>
#include <cuda_runtime.h>


#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = call; \
        if (err != cudaSuccess) { \
            fprintf(stderr, "CUDA Error at %s:%d - %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)


__global__ void findMaxMinKernel(int *d_in, int *d_min, int *d_max, int N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        int val = d_in[idx];


        atomicMin(d_min, val);


        atomicMax(d_max, val);
    }
}

int main() {
    int N;
    printf("Enter the size of array: ");
    scanf("%d", &N);


    int *h_a;
    int h_min = INT_MAX;
    int h_max = INT_MIN;

    int *d_a;
    int *d_min;
    int *d_max;

    h_a = (int*)malloc(N * sizeof(int));
    if (h_a == NULL) {
        fprintf(stderr, "Failed to allocate memory for h_a\n");
        return EXIT_FAILURE;
    }


    printf("Printing array with random values (0 - 20):\n");
    for (int i = 0; i < N; i++) {
        h_a[i] = rand() % 20;
    }
    for(int i=0;i<N;i++){
        printf("%d ",h_a[i]);
    }


    CUDA_CHECK(cudaMalloc((void**)&d_a, N * sizeof(int)));
    CUDA_CHECK(cudaMalloc((void**)&d_min, sizeof(int)));
    CUDA_CHECK(cudaMalloc((void**)&d_max, sizeof(int)));


    CUDA_CHECK(cudaMemcpy(d_min, &h_min, sizeof(int),
    cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_max, &h_max, sizeof(int),
    cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemcpy(d_a, h_a, N * sizeof(int),
    cudaMemcpyHostToDevice));

    int threadsPerBlock = 256;
    int numBlocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    findMaxMinKernel<<<numBlocks,
    threadsPerBlock>>>(d_a, d_min, d_max, N);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());


    CUDA_CHECK(cudaMemcpy(&h_min, d_min, sizeof(int),
    cudaMemcpyDeviceToHost));

    CUDA_CHECK(cudaMemcpy(&h_max, d_max, sizeof(int),
    cudaMemcpyDeviceToHost));


    printf("\nMin element: %d\n", h_min);
    printf("Max element: %d\n", h_max);


    free(h_a);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_min));
    CUDA_CHECK(cudaFree(d_max));

    return 0;
}


Overwriting max_min.cu


In [ ]:

!nvcc max_min.cu -o max_min
!./max_min


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Enter the size of array: 25
Printing array with random values (0 - 20):
3 6 17 15 13 15 6 12 9 1 2 7 10 19 3 6 0 6 12 16 11 8 7 9 2 
Min element: 0
Max element: 19
